<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/MiDashengLM-Gen_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<a href="https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/MiDashengLM-Gen_Colab.ipynb" target="_parent"><img src="https://img.shields.io/badge/GitHub-View_Source-181717?logo=github" alt="View on GitHub"/></a>

# 🤖 Xiaomi MiDashengLM-Gen & Dasheng-AudioGen — Unified LLM Audio Scene, Auto-Stop SFX & Talking OS Sound Pack Studio

<p align="left">
  <a href="https://github.com/xiaomi-research/midashenglm-gen"><img src="https://img.shields.io/badge/GitHub-xiaomi--research%2Fmidashenglm--gen-181717?logo=github" alt="GitHub"></a>
  <a href="https://huggingface.co/mispeech/midashenglm-gen"><img src="https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-mispeech%2Fmidashenglm--gen-amber" alt="HuggingFace"></a>
  <a href="https://huggingface.co/mispeech/Dasheng-AudioGen"><img src="https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Dasheng--AudioGen-orange" alt="Dasheng-AudioGen"></a>
  <a href="https://arxiv.org/abs/2608.11804"><img src="https://img.shields.io/badge/arXiv-2608.11804-b31b1b?logo=arxiv" alt="arXiv"></a>
  <img src="https://img.shields.io/badge/License-Apache%202.0%20(Ungated)-blue" alt="Apache-2.0">
</p>

Welcome to the **Xiaomi MiDashengLM-Gen & Dasheng-AudioGen** Google Colab studio! Developed by **Xiaomi Research (`mispeech`)**, this notebook integrates **both** of Xiaomi's open-weight unified audio scene generation models:

- **🧠 `MiDashengLM-Gen` (`mispeech/midashenglm-gen`)** — Combines a fine-tuned **Qwen3-1.7B** LLM backbone, `DashengTokenizer` (768-dim latents @ 25 Hz), a **10-step Euler ODE Flow-Matching DiT**, and a **learned binary Stop Head** (`stop_threshold`, `min_stop_step`) that automatically truncates output the instant a short sound effect finishes!
- **🎛️ `Dasheng-AudioGen` & `Dasheng-AudioGen-Multilingual`** — 25-step sway-sampled Flow-Matching model with `compose_prompt(caption, sfx, music, env, speech, asr)`.
- **🧩 Structured Multi-Aspect Scene Tagging** — Control `<|caption|>`, `<|sfx|>`, `<|music|>`, `<|env|>`, `<|speech|>`, and `<|asr|>` independently or simultaneously.
- **🤖 Talking Sci-Fi AI Assistant + SFX Windows Sound Pack Builder** — Because `MiDashengLM-Gen` synthesizes **intelligible speech + SFX + musical stingers simultaneously in a single pass**, you can generate Windows 10/11 Sound Schemes in either **Pure Instrumental/SFX Mode** OR **Talking AI Computer Mode** (e.g., a calm futuristic AI voice saying *"System online"* or *"Device connected"* blended seamlessly with a holographic telemetry chime!), auto-upsampled to **44.1 kHz / 48 kHz 16-bit PCM**.

In [ ]:
#@title Step 1: Verify GPU & Install MiDashengLM-Gen / Dasheng-AudioGen Dependencies
import os
import sys
import subprocess
import torch
from tqdm.auto import tqdm
from IPython.display import clear_output

def run_cmd(cmd, check=False):
    print(f"⏳ Running: {cmd}")
    return subprocess.run(cmd, shell=True, check=check)

try:
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print(f"✅ GPU Detected: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    else:
        print("⚠️ No CUDA GPU detected — GPU runtime is strongly recommended.")

    run_cmd("apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1", check=False)

    pip_pkgs = (
        'einops safetensors soundfile tqdm numpy x-transformers '
        '"transformers>=4.51.0,<5.0.0" huggingface_hub>=0.26.0 accelerate '
        'gradio>=5.0.0 matplotlib scipy librosa torchaudio'
    )
    run_cmd(f"{sys.executable} -m pip install -q -U {pip_pkgs}", check=False)

    clear_output()
    print("✅ Step 1 Complete: MiDashengLM-Gen & Dasheng-AudioGen dependencies installed!")
except Exception as e:
    print(f"❌ Installation Error: {e}")
    raise

In [ ]:
#@title Step 2: Mount Google Drive & Configure Persistent Cache
import os
import sys

WORKSPACE_DIR = "/content/midasheng_ws"
os.makedirs(WORKSPACE_DIR, exist_ok=True)

DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = os.path.ismount("/content/drive")
except Exception as e:
    print(f"ℹ️ Google Drive not mounted: {e}")

if DRIVE_MOUNTED:
    CACHE_ROOT = "/content/drive/MyDrive/AEI_Audio_Cache/midashenglm_gen"
    OUTPUT_DIR = "/content/drive/MyDrive/AEI_Audio_Exports/MiDashengLM_Gen"
else:
    CACHE_ROOT = os.path.join(WORKSPACE_DIR, "cache")
    OUTPUT_DIR = os.path.join(WORKSPACE_DIR, "outputs")

for d in (CACHE_ROOT, OUTPUT_DIR):
    os.makedirs(d, exist_ok=True)

os.environ["HF_HOME"] = os.path.join(CACHE_ROOT, "huggingface")
os.environ["TORCH_HOME"] = os.path.join(CACHE_ROOT, "torch")
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

print(f"📂 HF_HOME Cache : {os.environ['HF_HOME']}")
print(f"📂 Output Exports: {OUTPUT_DIR}")

In [ ]:
#@title Step 3: Core MiDashengLM-Gen & Dasheng-AudioGen Engine + Hi-Fi Upsampler
import os
import gc
import sys
import time
import glob
import scipy.signal
import torch
import torchaudio
import numpy as np
import soundfile as sf
from transformers import AutoModel

# ==============================================================================
# Shared Windows 10/11 Sound Scheme & Zero-Latency UI Mastering Engine
# ==============================================================================
import io
import re
import json
import uuid
import base64
import shutil
import zipfile
from datetime import datetime
import scipy.signal
import numpy as np
import soundfile as sf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

WINDOWS_SYSTEM_EVENTS = {
    "WindowsLogon": {
        "label": "Windows Logon / Startup",
        "reg_app": ".Default",
        "duration": 3.0,
        "contour": "warm welcoming ascending chord swell with a gentle shimmering reverb tail",
    },
    "WindowsLogoff": {
        "label": "Windows Logoff / Shutdown",
        "reg_app": ".Default",
        "duration": 2.5,
        "contour": "peaceful resolving descending harmonic chord that fades smoothly into silence",
    },
    "WindowsUnlock": {
        "label": "Windows Unlock / Sign-In",
        "reg_app": ".Default",
        "duration": 1.4,
        "contour": "bright welcoming two-note ascending unlock chime with airy resonance",
    },
    "SystemNotification": {
        "label": "System Notification (Toast)",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "crisp pleasant two-note notification chime, clear transient attack, short clean tail",
    },
    "WindowsNotifySystemGeneric": {
        "label": "Action Center / Generic Toast Banner",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "modern crystalline two-tone notification ping with soft harmonic decay",
    },
    "SystemAsterisk": {
        "label": "Asterisk / Info Pop-up",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "soft single crystalline ping with subtle harmonic resonance",
    },
    "SystemExclamation": {
        "label": "Exclamation / Warning Alert",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "attention-grabbing mid-range double pulse chime, alert but non-harsh",
    },
    "SystemHand": {
        "label": "Critical Stop / Error",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "low muted percussive double thud with a tense minor interval, error alert",
    },
    "SystemQuestion": {
        "label": "Question / Confirmation Dialog",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "gentle inquisitive rising two-note harmonic inflection chime",
    },
    "Default": {
        "label": "Default Beep / Fallback Alert",
        "reg_app": ".Default",
        "duration": 0.8,
        "contour": "clean short balanced harmonic ping, neutral interface alert",
    },
    "DeviceConnect": {
        "label": "Device Connect (USB Plug-In)",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "upbeat two-tone ascending interval (low note to higher fifth), hardware connected",
    },
    "DeviceDisconnect": {
        "label": "Device Disconnect (USB Unplug)",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "smooth two-tone descending interval (high note to lower root), hardware unplugged",
    },
    "DeviceFail": {
        "label": "Device Failed to Connect",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "short stuttering muted low buzz double tap, hardware connection failure",
    },
    "EmptyRecycleBin": {
        "label": "Empty Recycle Bin",
        "reg_app": "Explorer",
        "duration": 1.2,
        "contour": "crisp satisfying paper and glass shimmer dissolve whoosh",
    },
    "WindowsUAC": {
        "label": "User Account Control (UAC Prompt)",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "authoritative clean two-note security confirmation chime",
    },
    "LowBatteryAlarm": {
        "label": "Low Battery Alarm",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "gentle pulsing two-note reminder tone",
    },
    "CriticalBatteryAlarm": {
        "label": "Critical Battery Alarm",
        "reg_app": ".Default",
        "duration": 1.5,
        "contour": "urgent rapid triple pulse warning chime",
    },
    "MailBeep": {
        "label": "New Mail / Message Beep",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "bright airy bell ding notification",
    },
    "MessageNudge": {
        "label": "Message Nudge / Chat Ping",
        "reg_app": ".Default",
        "duration": 0.9,
        "contour": "playful double tactile pop ping notification",
    },
    "PrintComplete": {
        "label": "Print / Task Complete",
        "reg_app": ".Default",
        "duration": 1.3,
        "contour": "satisfying upbeat three-note completion flourish chime",
    },
    "MenuCommand": {
        "label": "Menu Command / Tactile UI Click",
        "reg_app": ".Default",
        "duration": 0.6,
        "contour": "ultra-short crisp tactile UI click pop transient, dry and immediate",
    },
}

SONIC_THEME_PRESETS = {
    "Frutiger Aero Crystal Glass": (
        "crystalline water-drop glass marimba and airy harmonic bell pads, "
        "clean high-tech Frutiger Aero operating system UI sound effect, studio quality"
    ),
    "Cyberpunk Neon HUD": (
        "futuristic cyberpunk holographic HUD telemetry synthesizer, "
        "crisp digital micro-transient with warm sub-bass resonance, sci-fi UI sound"
    ),
    "Cozy Wooden Kalimba & Mallet": (
        "warm organic rosewood marimba, thumb piano kalimba, and soft felt mallet percussion, "
        "intimate close-mic acoustic UI sound effect"
    ),
    "Sci-Fi Spaceship Bridge": (
        "clean starship computer console telemetry bleep, precision FM synthesis harmonic tone, "
        "futuristic spacecraft interface sound effect"
    ),
    "Retro 16-Bit Chiptune Arcade": (
        "crisp nostalgic 16-bit FM synth arcade game UI sound effect, "
        "bright square and triangle wave arpeggio, clean retro console audio"
    ),
    "Warm Lo-Fi Analog Tape Synth": (
        "warm vintage analog poly-synth and Rhodes electric piano chime with gentle tape saturation, "
        "cozy lo-fi interface sound effect"
    ),
    "Minimalist Tactile Micro-Foley": (
        "ultra-clean modern minimalist UI micro-foley, soft mechanical switch click and muted glass tap, "
        "tight dry studio recording"
    ),
    "Dark Fantasy Crystal & Harp": (
        "ethereal celesta, delicate harp harmonic pluck, and mystical resonant crystal chime, "
        "fantasy RPG interface sound effect"
    ),
}

DEFAULT_SELECTED_WIN_EVENTS = [
    "WindowsLogon",
    "WindowsLogoff",
    "WindowsUnlock",
    "SystemNotification",
    "WindowsNotifySystemGeneric",
    "SystemAsterisk",
    "SystemExclamation",
    "SystemHand",
    "DeviceConnect",
    "DeviceDisconnect",
    "EmptyRecycleBin",
    "WindowsUAC",
]


def build_windows_event_prompt(event_key: str, theme_name: str, custom_style: str = "") -> str:
    ev = WINDOWS_SYSTEM_EVENTS.get(event_key, WINDOWS_SYSTEM_EVENTS["SystemNotification"])
    style_desc = custom_style.strip() if custom_style and custom_style.strip() else SONIC_THEME_PRESETS.get(
        theme_name, SONIC_THEME_PRESETS["Frutiger Aero Crystal Glass"]
    )
    return f"{ev['contour']}, {style_desc}, high fidelity, no background noise"


def apply_phase_safe_stereo_widener(arr: np.ndarray, sr: int, width: float = 0.22) -> np.ndarray:
    """Upgrade mono (N, 1) to phase-coherent stereo (N, 2) or subtly widen stereo above 350 Hz."""
    if arr.ndim == 1:
        arr = arr[:, None]
    n_samples, ch = arr.shape
    if n_samples < 64 or width <= 0.01:
        return np.repeat(arr, 2, axis=1) if ch == 1 else arr

    nyq = 0.5 * float(sr)
    cutoff = min(max(350.0 / nyq, 0.01), 0.85)
    b, a = scipy.signal.butter(2, cutoff, btype="highpass")

    if ch == 1:
        mono = arr[:, 0]
        high = scipy.signal.lfilter(b, a, mono).astype(np.float32)
        delay_samples = max(1, int(0.00038 * sr))  # ~0.38 ms Haas micro-decorrelation
        side = np.zeros_like(high)
        if delay_samples < n_samples:
            side[delay_samples:] = high[:-delay_samples] - high[delay_samples:]
        side *= float(width)
        left = mono + side
        right = mono - side
        return np.stack([left, right], axis=1).astype(np.float32)
    else:
        mid = 0.5 * (arr[:, 0] + arr[:, 1])
        side = 0.5 * (arr[:, 0] - arr[:, 1])
        side_high = scipy.signal.lfilter(b, a, side).astype(np.float32)
        side_boosted = side + float(width) * 0.5 * side_high
        return np.stack([mid + side_boosted, mid - side_boosted], axis=1).astype(np.float32)


def master_ui_sound(
    audio_np: np.ndarray,
    sr: int,
    target_duration_s: float = None,
    trim_silence: bool = True,
    trim_db: float = -45.0,
    fade_in_ms: float = 1.5,
    fade_out_ms: float = 120.0,
    target_dbfs: float = -1.2,
    stereo_widen: bool = True,
    hpf_hz: float = 40.0,
    pitch_semitones: float = 0.0,
) -> np.ndarray:
    """Zero-latency UI sound mastering: snaps to first transient (<1ms pre-roll), 40Hz sub-rumble filter, optional semitone transpose, 1.5ms click-free micro-fade, phase-safe stereo widening, and true-peak normalization."""
    arr = np.asarray(audio_np, dtype=np.float32)
    if arr.ndim == 1:
        arr = arr[:, None]  # (samples, 1)
    elif arr.ndim == 2 and arr.shape[0] <= 2 and arr.shape[1] > arr.shape[0]:
        arr = arr.T  # (samples, channels)

    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    arr = arr - np.mean(arr, axis=0, keepdims=True)

    # Optional sub-rumble high-pass filter (default 40 Hz) to keep UI chimes crisp
    if hpf_hz and float(hpf_hz) > 10.0 and arr.shape[0] > 64:
        nyq = 0.5 * float(sr)
        cutoff = min(max(float(hpf_hz) / nyq, 0.002), 0.45)
        b_hp, a_hp = scipy.signal.butter(2, cutoff, btype="highpass")
        arr = scipy.signal.lfilter(b_hp, a_hp, arr, axis=0).astype(np.float32)

    # Optional semitone pitch shift via high-precision polyphase resampling
    if pitch_semitones and abs(float(pitch_semitones)) >= 0.05 and arr.shape[0] > 64:
        factor = 2.0 ** (float(pitch_semitones) / 12.0)
        up = 1000
        down = max(100, int(round(1000.0 * factor)))
        gcd = np.gcd(up, down)
        arr = scipy.signal.resample_poly(arr, up // gcd, down // gcd, axis=0).astype(np.float32)

    if trim_silence and arr.shape[0] > int(0.04 * sr):
        mono_env = np.max(np.abs(arr), axis=1)
        diff_env = np.abs(np.diff(mono_env, prepend=mono_env[0]))
        peak = float(np.max(mono_env))
        if peak > 1e-6:
            amp_thresh = peak * (10.0 ** (float(trim_db) / 20.0))
            flux_thresh = float(np.max(diff_env)) * 0.08
            active_idx = np.where((mono_env >= amp_thresh) | (diff_env >= flux_thresh))[0]
            if active_idx.size > 0:
                pre_roll = int(0.0008 * sr)   # 0.8 ms zero-latency pre-roll
                post_roll = int(0.160 * sr)   # 160 ms natural reverb tail retention
                start_i = max(0, int(active_idx[0]) - pre_roll)
                end_i = min(arr.shape[0], int(active_idx[-1]) + post_roll)
                if end_i > start_i + int(0.03 * sr):
                    arr = arr[start_i:end_i]

    if target_duration_s is not None and target_duration_s > 0:
        max_samples = int(float(target_duration_s) * sr)
        if arr.shape[0] > max_samples:
            arr = arr[:max_samples]

    n_samples = arr.shape[0]
    if n_samples == 0:
        return np.zeros((int(0.5 * sr), 2), dtype=np.float32)

    if stereo_widen:
        arr = apply_phase_safe_stereo_widener(arr, sr, width=0.22 if arr.shape[1] == 1 else 0.12)

    fi_samples = min(int((float(fade_in_ms) / 1000.0) * sr), max(1, n_samples // 8))
    if fi_samples > 1:
        fi_curve = 0.5 * (1.0 - np.cos(np.linspace(0.0, np.pi, fi_samples, dtype=np.float32)))
        arr[:fi_samples] *= fi_curve[:, None]

    fo_samples = min(int((float(fade_out_ms) / 1000.0) * sr), max(1, n_samples // 2))
    if fo_samples > 1:
        fo_curve = (0.5 * (1.0 + np.cos(np.linspace(0.0, np.pi, fo_samples, dtype=np.float32)))) ** 1.2
        arr[-fo_samples:] *= fo_curve[:, None]

    peak_val = float(np.max(np.abs(arr)))
    if peak_val > 1e-6:
        target_amp = 10.0 ** (float(target_dbfs) / 20.0)
        arr = arr * (target_amp / peak_val)

    arr = np.clip(arr, -0.999, 0.999)
    return arr if arr.shape[1] > 1 else arr[:, 0]


def save_pcm16_wav(path: str, audio_np: np.ndarray, sr: int, subtype: str = "PCM_16") -> str:
    abs_path = os.path.abspath(path)
    os.makedirs(os.path.dirname(abs_path), exist_ok=True)
    sf.write(abs_path, audio_np, int(sr), subtype=subtype)
    # Mirror to AEI_TTS_Out if Google Drive is mounted so companion notebooks (InfiniteTalk, SeedVR2, Audio_PostProcessor) auto-discover it
    if "/content/drive/MyDrive/" in abs_path and "/AEI_TTS_Out/" not in abs_path:
        try:
            mirror_dir = "/content/drive/MyDrive/AEI_TTS_Out/SFX_Exports"
            os.makedirs(mirror_dir, exist_ok=True)
            shutil.copy2(abs_path, os.path.join(mirror_dir, os.path.basename(abs_path)))
        except Exception:
            pass
    return abs_path


def load_audio_file_safe(audio_path: str, target_sr: int = None, mono: bool = False):
    """Robust audio loader using soundfile first, falling back to librosa/torchaudio."""
    try:
        data, sr = sf.read(audio_path, dtype="float32", always_2d=True)
        # data is (T, C) -> transpose to (C, T)
        wav = data.T
    except Exception:
        import librosa
        wav, sr = librosa.load(audio_path, sr=None, mono=False)
        if wav.ndim == 1:
            wav = wav[None, :]
    if mono and wav.shape[0] > 1:
        wav = np.mean(wav, axis=0, keepdims=True)
    if target_sr is not None and int(sr) != int(target_sr):
        gcd = np.gcd(int(sr), int(target_sr))
        wav = scipy.signal.resample_poly(wav, int(target_sr) // gcd, int(sr) // gcd, axis=-1).astype(np.float32)
        sr = int(target_sr)
    return wav, int(sr)


def render_waveform_and_spectrogram(wav_path: str, title: str = "Audio Signal & Spectrogram") -> str:
    """Render a dual-panel waveform + spectrogram plot for visual transient & tail inspection."""
    try:
        data, sr = sf.read(wav_path, always_2d=True)
        mono = np.mean(data, axis=1)
        dur = len(mono) / float(sr)
        t = np.linspace(0.0, dur, len(mono))

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8.5, 4.2), facecolor="#0f172a")
        for ax in (ax1, ax2):
            ax.set_facecolor("#1e293b")
            ax.tick_params(colors="#cbd5e1", labelsize=8)
            for spine in ax.spines.values():
                spine.set_color("#334155")

        if data.shape[1] > 1:
            ax1.plot(t, data[:, 0], color="#38bdf8", linewidth=0.8, alpha=0.85, label="L")
            ax1.plot(t, data[:, 1], color="#f472b6", linewidth=0.8, alpha=0.75, label="R")
            ax1.legend(loc="upper right", fontsize=7, facecolor="#0f172a", edgecolor="#334155", labelcolor="#f8fafc")
        else:
            ax1.plot(t, mono, color="#38bdf8", linewidth=0.9)
        ax1.set_xlim(0, max(dur, 0.05))
        ax1.set_ylim(-1.05, 1.05)
        ax1.set_ylabel("Amplitude", color="#e2e8f0", fontsize=8)
        ch_label = "Stereo" if data.shape[1] > 1 else "Mono"
        ax1.set_title(f"{title} ({sr} Hz {ch_label}, {dur:.2f}s)", color="#f8fafc", fontsize=10, fontweight="bold")
        ax1.grid(True, color="#334155", alpha=0.4, linestyle="--")

        nfft = min(1024, max(128, 2 ** int(np.log2(max(len(mono) // 8, 128)))))
        ax2.specgram(mono + 1e-10, NFFT=nfft, Fs=sr, noverlap=nfft // 2, cmap="magma")
        ax2.set_xlim(0, max(dur, 0.05))
        ax2.set_xlabel("Time (seconds)", color="#e2e8f0", fontsize=8)
        ax2.set_ylabel("Freq (Hz)", color="#e2e8f0", fontsize=8)

        plt.tight_layout()
        out_png = os.path.join("/tmp", f"spec_{uuid.uuid4().hex[:8]}.png")
        fig.savefig(out_png, dpi=120, facecolor=fig.get_facecolor())
        plt.close(fig)
        return out_png
    except Exception as e:
        print(f"Spectrogram render warning: {e}")
        return None


def build_audition_soundboard_html(
    scheme_name: str,
    event_wav_map: dict,
    theme_name: str,
    model_label: str,
    embed_base64: bool = False,
) -> str:
    """Build a dark-mode HTML5 WebAudio interactive soundboard with 1-click sequential Play-All auditioning."""
    board_uid = f"sb_{uuid.uuid4().hex[:8]}"
    cards_html = []
    for ev_key, wav_info in event_wav_map.items():
        ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, {"label": ev_key, "reg_app": ".Default", "duration": 1.0})
        if embed_base64 and os.path.exists(wav_info["path"]):
            with open(wav_info["path"], "rb") as bf:
                b64 = base64.b64encode(bf.read()).decode("ascii")
            audio_src = f"data:audio/wav;base64,{b64}"
        else:
            audio_src = f"Media/{wav_info['filename']}"

        cards_html.append(
            f'<div class="aei-sb-card" style="background:#1e293b;border:1px solid #334155;border-radius:10px;padding:10px 12px;display:flex;flex-direction:column;gap:6px;transition:all 0.2s ease;">'
            f'<div style="display:flex;justify-content:space-between;align-items:center;">'
            f'<strong style="color:#f8fafc;font-size:13px;">🔔 {ev_meta.get("label", ev_key)}</strong>'
            f'<span style="background:#0f172a;color:#38bdf8;font-size:11px;padding:2px 6px;border-radius:4px;font-family:monospace;">{ev_meta.get("reg_app", ".Default")}\\{ev_key}</span>'
            f'</div>'
            f'<audio controls preload="none" src="{audio_src}" style="width:100%;height:32px;"></audio>'
            f'</div>'
        )
    grid_str = "\n".join(cards_html)
    play_all_js = (
        f"const root=document.getElementById('{board_uid}');"
        f"if(!root)return;"
        f"const audios=Array.from(root.querySelectorAll('audio'));"
        f"const cards=Array.from(root.querySelectorAll('.aei-sb-card'));"
        f"if(root._playing){{root._playing=false;audios.forEach(a=>{{a.pause();a.currentTime=0;}});cards.forEach(c=>c.style.borderColor='#334155');this.textContent='▶ Play All Events Sequentially';return;}}"
        f"root._playing=true;this.textContent='⏹ Stop Sequence';"
        f"let idx=0;const btn=this;"
        f"const next=()=>{{if(!root._playing||idx>=audios.length){{root._playing=false;btn.textContent='▶ Play All Events Sequentially';cards.forEach(c=>c.style.borderColor='#334155');return;}}"
        f"cards.forEach((c,i)=>{{c.style.borderColor=(i===idx)?'#38bdf8':'#334155';}});"
        f"const a=audios[idx++];a.currentTime=0;"
        f"a.onended=()=>{{setTimeout(next,280);}};"
        f"a.onerror=()=>{{setTimeout(next,100);}};"
        f"a.play().catch(()=>setTimeout(next,100));}};"
        f"next();"
    )
    return (
        f'<div id="{board_uid}" style="background:#0f172a;color:#f8fafc;padding:16px;border-radius:12px;border:1px solid #334155;font-family:system-ui,-apple-system,sans-serif;">'
        f'<div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:10px;margin-bottom:12px;">'
        f'<div>'
        f'<h3 style="margin:0 0 4px 0;color:#38bdf8;font-size:16px;">🪟 {scheme_name} — Interactive Windows 10/11 Soundboard</h3>'
        f'<p style="margin:0;color:#94a3b8;font-size:12px;">Theme: <b>{theme_name}</b> · Engine: <b>{model_label}</b> · Events: <b>{len(event_wav_map)}</b></p>'
        f'</div>'
        f'<button type="button" onclick="{play_all_js}" style="background:#0284c7;color:#ffffff;border:none;border-radius:8px;padding:8px 14px;font-size:12px;font-weight:600;cursor:pointer;">▶ Play All Events Sequentially</button>'
        f'</div>'
        f'<div style="display:grid;grid-template-columns:repeat(auto-fill,minmax(270px,1fr));gap:10px;">'
        f'{grid_str}'
        f'</div>'
        f'</div>'
    )


LAST_WINPACK_STATE = {
    "pack_dir": None,
    "scheme_name": "AEI_Custom_Scheme",
    "theme_name": "Frutiger Aero Crystal Glass",
    "model_label": "AEI Sound Studio",
    "output_dir": "/content",
    "event_wav_map": {},
}


def package_windows_sound_scheme_zip(
    scheme_name: str,
    event_wav_map: dict,
    theme_name: str,
    model_label: str,
    output_dir: str,
) -> str:
    """Create a complete Windows 10/11 Sound Scheme ZIP with .wav files, Install/Uninstall .ps1, .reg file, Scheme.ini, soundpack_manifest.json, and audition_board.html."""
    safe_id = re.sub(r"[^A-Za-z0-9_-]+", "_", (scheme_name or "AEI_Custom_Sounds").strip())[:32].strip("_") or "AEI_Custom"
    display_name = (scheme_name or "AEI Custom Sounds").strip()[:48]
    os.makedirs(output_dir, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    zip_path = os.path.join(output_dir, f"{safe_id}_Windows_SoundPack_{ts}.zip")

    ps1_lines = [
        f"# ==============================================================================",
        f"# Windows 10/11 Sound Scheme Automated Installer: {display_name}",
        f"# Generated by AEI Colab Notebooks ({model_label})",
        f"# ==============================================================================",
        f"$ErrorActionPreference = 'Stop'",
        f"$SchemeId = '{safe_id}'",
        f"$SchemeDisplay = '{display_name}'",
        f"$TargetDir = Join-Path $env:LOCALAPPDATA \"WindowsSoundSchemes\\$SchemeId\"",
        f"New-Item -ItemType Directory -Force -Path $TargetDir | Out-Null",
        f"$BackupReg = Join-Path $TargetDir 'Backup_Previous_AppEvents.reg'",
        f"if (-not (Test-Path $BackupReg)) {{ reg export 'HKCU\\AppEvents\\Schemes' $BackupReg /y | Out-Null }}",
        f"Copy-Item -Path (Join-Path $PSScriptRoot 'Media\\*.wav') -Destination $TargetDir -Force",
        f"",
        f"$NamesKey = \"HKCU:\\AppEvents\\Schemes\\Names\\$SchemeId\"",
        f"New-Item -Path $NamesKey -Force | Out-Null",
        f"Set-ItemProperty -Path $NamesKey -Name '(Default)' -Value $SchemeDisplay",
        f"",
        f"$EventMap = @{{",
    ]
    for ev_key, wav_info in event_wav_map.items():
        ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, {"reg_app": ".Default"})
        ps1_lines.append(f"    '{ev_meta['reg_app']}\\{ev_key}' = '{wav_info['filename']}'")
    ps1_lines += [
        f"}}",
        f"$FirstWav = $null",
        f"foreach ($entry in $EventMap.GetEnumerator()) {{",
        f"    $wavPath = Join-Path $TargetDir $entry.Value",
        f"    if (-not $FirstWav -and (Test-Path $wavPath)) {{ $FirstWav = $wavPath }}",
        f"    $schemeEventKey = \"HKCU:\\AppEvents\\Schemes\\Apps\\$($entry.Key)\\$SchemeId\"",
        f"    $currentEventKey = \"HKCU:\\AppEvents\\Schemes\\Apps\\$($entry.Key)\\.Current\"",
        f"    New-Item -Path $schemeEventKey -Force | Out-Null",
        f"    Set-ItemProperty -Path $schemeEventKey -Name '(Default)' -Value $wavPath",
        f"    New-Item -Path $currentEventKey -Force | Out-Null",
        f"    Set-ItemProperty -Path $currentEventKey -Name '(Default)' -Value $wavPath",
        f"}}",
        f"Set-ItemProperty -Path 'HKCU:\\AppEvents\\Schemes' -Name '(Default)' -Value $SchemeId",
        f"if ($FirstWav) {{ try {{ (New-Object System.Media.SoundPlayer $FirstWav).Play() }} catch {{}} }}",
        f"Write-Host \"Successfully installed and activated Windows Sound Scheme: $SchemeDisplay\" -ForegroundColor Green",
        f"Write-Host \"Prior registry state backed up to: $BackupReg\" -ForegroundColor Cyan",
        f"Start-Process 'control.exe' -ArgumentList 'mmsys.cpl,,2'",
    ]

    uninstall_ps1 = [
        f"# ==============================================================================",
        f"# Windows 10/11 Sound Scheme Uninstaller & Default Restore: {display_name}",
        f"# ==============================================================================",
        f"$ErrorActionPreference = 'SilentlyContinue'",
        f"$SchemeId = '{safe_id}'",
        f"Set-ItemProperty -Path 'HKCU:\\AppEvents\\Schemes' -Name '(Default)' -Value '.Default'",
        f"Get-ChildItem 'HKCU:\\AppEvents\\Schemes\\Apps' -Recurse | Where-Object {{ $_.PSChildName -eq $SchemeId }} | Remove-Item -Recurse -Force",
        f"Remove-Item -Path \"HKCU:\\AppEvents\\Schemes\\Names\\$SchemeId\" -Recurse -Force",
        f"$TargetDir = Join-Path $env:LOCALAPPDATA \"WindowsSoundSchemes\\$SchemeId\"",
        f"if (Test-Path $TargetDir) {{ Remove-Item -Path $TargetDir -Recurse -Force }}",
        f"Write-Host \"Restored Windows Default sound scheme and removed $SchemeId.\" -ForegroundColor Yellow",
        f"Start-Process 'control.exe' -ArgumentList 'mmsys.cpl,,2'",
    ]

    reg_lines = [
        "Windows Registry Editor Version 5.00",
        "",
        f"; Windows Sound Scheme: {display_name} ({model_label})",
        f"[HKEY_CURRENT_USER\\AppEvents\\Schemes\\Names\\{safe_id}]",
        f"@=\"{display_name}\"",
        "",
    ]
    for ev_key, wav_info in event_wav_map.items():
        ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, {"reg_app": ".Default"})
        escaped_path = f"C:\\\\Windows\\\\Media\\\\{safe_id}\\\\{wav_info['filename']}"
        reg_lines += [
            f"[HKEY_CURRENT_USER\\AppEvents\\Schemes\\Apps\\{ev_meta['reg_app']}\\{ev_key}\\{safe_id}]",
            f"@=\"{escaped_path}\"",
            "",
            f"[HKEY_CURRENT_USER\\AppEvents\\Schemes\\Apps\\{ev_meta['reg_app']}\\{ev_key}\\.Current]",
            f"@=\"{escaped_path}\"",
            "",
        ]

    ini_lines = [
        "[Scheme]",
        f"Name={display_name}",
        f"SchemeId={safe_id}",
        f"Theme={theme_name}",
        f"Generator={model_label}",
        f"CreatedAt={datetime.now().isoformat()}",
        "",
        "[Events]",
    ]
    for ev_key, wav_info in event_wav_map.items():
        ini_lines.append(f"{ev_key}=Media/{wav_info['filename']}")
        ini_lines.append(f"{ev_key}_Prompt={wav_info.get('prompt', '')}")

    manifest_data = {
        "scheme_name": display_name,
        "scheme_id": safe_id,
        "theme_name": theme_name,
        "generator": model_label,
        "created_at": datetime.now().isoformat(),
        "events": {
            k: {
                "filename": f"Media/{v['filename']}",
                "label": WINDOWS_SYSTEM_EVENTS.get(k, {}).get("label", k),
                "reg_app": WINDOWS_SYSTEM_EVENTS.get(k, {}).get("reg_app", ".Default"),
                "prompt": v.get("prompt", ""),
            }
            for k, v in event_wav_map.items()
        },
    }

    standalone_html = (
        "<!DOCTYPE html>\n<html><head><meta charset='utf-8'>"
        f"<title>{display_name} — Windows Soundboard</title></head>"
        "<body style='background:#020617;margin:24px;'>"
        + build_audition_soundboard_html(display_name, event_wav_map, theme_name, model_label, embed_base64=False)
        + "</body></html>"
    )

    readme_txt = (
        f"====================================================================\n"
        f"  {display_name} — Windows 10 / 11 Sound Event Pack\n"
        f"  Generated with: {model_label}\n"
        f"  Theme Preset:   {theme_name}\n"
        f"====================================================================\n\n"
        f"PREVIEW IN BROWSER FIRST:\n"
        f"  Open 'audition_board.html' in any browser to audition all {len(event_wav_map)} events!\n\n"
        f"OPTION 1: 1-Click PowerShell Install (Recommended, No Admin Required)\n"
        f"--------------------------------------------------------------------\n"
        f"1. Extract this entire ZIP archive to any folder.\n"
        f"2. Right-click 'Install_{safe_id}.ps1' -> 'Run with PowerShell'.\n"
        f"3. The script copies the .wav files to %LOCALAPPDATA%\\WindowsSoundSchemes\\{safe_id},\n"
        f"   registers '{display_name}' in the Windows Sound Control Panel, activates it,\n"
        f"   and opens mmsys.cpl so you can preview or customize individual events!\n"
        f"4. To revert at any time, right-click 'Uninstall_{safe_id}.ps1' -> 'Run with PowerShell'.\n\n"
        f"OPTION 2: Manual Registry (.reg) Import\n"
        f"--------------------------------------------------------------------\n"
        f"1. Copy the 'Media' folder contents into 'C:\\Windows\\Media\\{safe_id}\\'.\n"
        f"2. Double-click 'Install_{safe_id}.reg' to register the sound scheme.\n"
    )

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for ev_key, wav_info in event_wav_map.items():
            zf.write(wav_info["path"], arcname=f"Media/{wav_info['filename']}")
        zf.writestr(f"Install_{safe_id}.ps1", "\r\n".join(ps1_lines))
        zf.writestr(f"Uninstall_{safe_id}.ps1", "\r\n".join(uninstall_ps1))
        zf.writestr(f"Install_{safe_id}.reg", "\r\n".join(reg_lines))
        zf.writestr("Scheme.ini", "\r\n".join(ini_lines))
        zf.writestr("soundpack_manifest.json", json.dumps(manifest_data, indent=2))
        zf.writestr("audition_board.html", standalone_html)
        zf.writestr("README_INSTALL.txt", readme_txt.replace("\n", "\r\n"))

    if event_wav_map:
        first_entry = next(iter(event_wav_map.values()))
        if isinstance(first_entry, dict) and "path" in first_entry:
            LAST_WINPACK_STATE["pack_dir"] = os.path.dirname(first_entry["path"])
    LAST_WINPACK_STATE["scheme_name"] = display_name
    LAST_WINPACK_STATE["theme_name"] = theme_name
    LAST_WINPACK_STATE["model_label"] = model_label
    LAST_WINPACK_STATE["output_dir"] = output_dir
    LAST_WINPACK_STATE["event_wav_map"] = dict(event_wav_map)
    return zip_path


def finalize_rerolled_event_in_pack(
    ev_key: str,
    mastered_wav: np.ndarray,
    sr: int,
    ev_prompt: str,
    output_dir: str,
    scheme_name: str = "",
    theme_name: str = "",
    model_label: str = "",
):
    """Save a re-rolled single Windows event .wav into the active pack state, re-bundle the ZIP, and refresh the soundboard."""
    pack_dir = LAST_WINPACK_STATE.get("pack_dir")
    if not pack_dir or not os.path.exists(pack_dir):
        pack_dir = os.path.join(output_dir, f"winpack_reroll_{int(datetime.now().timestamp())}")
        os.makedirs(pack_dir, exist_ok=True)
        LAST_WINPACK_STATE["pack_dir"] = pack_dir

    fname = f"{ev_key}.wav"
    fpath = os.path.join(pack_dir, fname)
    save_pcm16_wav(fpath, mastered_wav, sr)

    ev_map = dict(LAST_WINPACK_STATE.get("event_wav_map") or {})
    ev_map[ev_key] = {"path": fpath, "filename": fname, "prompt": ev_prompt}
    LAST_WINPACK_STATE["event_wav_map"] = ev_map

    s_name = (scheme_name or LAST_WINPACK_STATE.get("scheme_name") or "AEI_Custom_Scheme").strip()
    t_name = (theme_name or LAST_WINPACK_STATE.get("theme_name") or "Custom Theme").strip()
    m_lbl = (model_label or LAST_WINPACK_STATE.get("model_label") or "AEI Sound Studio").strip()

    zip_path = package_windows_sound_scheme_zip(
        scheme_name=s_name,
        event_wav_map=ev_map,
        theme_name=t_name,
        model_label=m_lbl,
        output_dir=output_dir,
    )
    LAST_WINPACK_STATE["pack_dir"] = pack_dir
    board_html = build_audition_soundboard_html(
        scheme_name=s_name,
        event_wav_map=ev_map,
        theme_name=t_name,
        model_label=m_lbl,
        embed_base64=True,
    )
    spec_img = render_waveform_and_spectrogram(fpath, title=f"Re-Rolled Event: {ev_key}")
    status = (
        f"🔄 Re-rolled **`{ev_key}`** ({WINDOWS_SYSTEM_EVENTS.get(ev_key, {}).get('label', ev_key)}) and updated "
        f"**`{os.path.basename(zip_path)}`** ({len(ev_map)} total events in pack)!"
    )
    return zip_path, fpath, spec_img, board_html, status


TALKING_OS_PHRASES = {
    "WindowsLogon": "System online. Welcome back.",
    "WindowsLogoff": "Shutting down system services. Goodbye.",
    "SystemNotification": "New system notification received.",
    "SystemAsterisk": "Information update.",
    "SystemExclamation": "Attention required.",
    "SystemHand": "Critical system error detected.",
    "DeviceConnect": "External hardware connected.",
    "DeviceDisconnect": "Hardware device disconnected.",
    "DeviceFail": "Hardware connection failed.",
    "EmptyRecycleBin": "Recycle bin cleared.",
    "WindowsUAC": "Authorization required.",
    "LowBatteryAlarm": "Warning: battery level is low.",
    "CriticalBatteryAlarm": "Alert: critical battery level.",
    "MailBeep": "Incoming message.",
    "PrintComplete": "Task completed successfully.",
    "MenuCommand": "",
    "WindowsNotifySystemGeneric": "System notification.",
    "WindowsUnlock": "Workstation unlocked.",
    "SystemQuestion": "Confirmation requested.",
    "Default": "",
    "MessageNudge": "Attention please.",
}


def _extract_dasheng_audio(raw) -> np.ndarray:
    """Safely convert torch.Tensor, list of tensors, or numpy array output from MiDashengLM-Gen into 1D float32 numpy."""
    if isinstance(raw, (list, tuple)) and len(raw) > 0:
        raw = raw[0]
    if isinstance(raw, torch.Tensor):
        raw = raw.detach().cpu().float().numpy()
    return np.asarray(raw, dtype=np.float32).squeeze()


def upsample_with_harmonic_air(audio_np: np.ndarray, in_sr: int, target_sr: int = 44100, air_boost: float = 0.08) -> tuple:
    """Polyphase resample 16 kHz audio to 44.1 / 48 kHz with subtle high-frequency harmonic exciter."""
    arr = np.asarray(audio_np, dtype=np.float32).squeeze()
    if int(in_sr) == int(target_sr):
        return arr, int(target_sr)
    gcd = np.gcd(int(in_sr), int(target_sr))
    up = int(target_sr) // gcd
    down = int(in_sr) // gcd
    resampled = scipy.signal.resample_poly(arr, up, down).astype(np.float32)

    if air_boost > 0.001 and len(resampled) > 64:
        # Soft tanh harmonic excitation filtered above 6.5 kHz
        nyq = 0.5 * target_sr
        cutoff = min(6500.0 / nyq, 0.85)
        b, a = scipy.signal.butter(2, cutoff, btype="highpass")
        high_harmonics = scipy.signal.lfilter(b, a, np.tanh(resampled * 2.5))
        resampled = resampled + float(air_boost) * high_harmonics.astype(np.float32)

    return resampled, int(target_sr)


def format_midasheng_tagged_prompt(
    caption: str,
    sfx: str = "",
    music: str = "",
    env: str = "",
    speech: str = "",
    asr: str = "",
    for_audiogen: bool = False,
) -> str:
    cap = (caption or sfx or "High quality sound effect").strip()
    if for_audiogen:
        parts = [f"<|caption|> {cap}"]
        if speech and speech.strip():
            parts.append(f"<|speech|> {speech.strip()}")
        if asr and asr.strip():
            parts.append(f"<|asr|> {asr.strip()}")
        if sfx and sfx.strip():
            parts.append(f"<|sfx|> {sfx.strip()}")
        if music and music.strip():
            parts.append(f"<|music|> {music.strip()}")
        if env and env.strip():
            parts.append(f"<|env|> {env.strip()}")
        return " ".join(parts)
    else:
        return (
            f"<|caption|> {cap} "
            f"<|asr|> {asr.strip() if (asr and asr.strip()) else '<|unknown|>'} "
            f"<|speech|> {speech.strip() if (speech and speech.strip()) else '<|unknown|>'} "
            f"<|music|> {music.strip() if (music and music.strip()) else '<|unknown|>'} "
            f"<|sfx|> {sfx.strip() if (sfx and sfx.strip()) else '<|unknown|>'} "
            f"<|env|> {env.strip() if (env and env.strip()) else '<|unknown|>'}"
        )


class XiaomiDashengStudio:
    """Unified manager for mispeech/midashenglm-gen and mispeech/Dasheng-AudioGen."""

    def __init__(self):
        self.model = None
        self.current_repo = None
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

    def unload(self):
        self.model = None
        self.current_repo = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def ensure_loaded(self, engine_choice: str) -> str:
        repo_id = engine_choice.split(" ")[0].strip()
        if self.current_repo == repo_id and self.model is not None:
            return f"✅ Using cached `{repo_id}` on `{self.device}`."
        self.unload()
        self.model = AutoModel.from_pretrained(repo_id, trust_remote_code=True)
        if self.device == "cuda":
            self.model = self.model.cuda()
        self.model.eval()
        self.current_repo = repo_id
        return f"✅ Loaded `{repo_id}` on `{self.device}`."

    @torch.inference_mode()
    def generate_scene(
        self,
        engine_choice: str,
        caption: str,
        sfx: str = "",
        music: str = "",
        env: str = "",
        speech: str = "",
        asr: str = "",
        eval_cfg: float = 2.5,
        stop_threshold: float = 0.5,
        min_stop_step: int = 5,
        num_steps: int = 25,
        sway_coef: float = -1.0,
        seed: int = 42,
        target_sr: int = 44100,
    ):
        msg = self.ensure_loaded(engine_choice)
        actual_seed = int(seed) if int(seed) >= 0 else int(np.random.randint(1, 999999))
        torch.manual_seed(actual_seed)

        is_audiogen = "Dasheng-AudioGen" in self.current_repo
        tagged = format_midasheng_tagged_prompt(
            caption=caption, sfx=sfx, music=music, env=env, speech=speech, asr=asr, for_audiogen=is_audiogen
        )

        if is_audiogen:
            if hasattr(self.model, "compose_prompt"):
                kw = {"caption": (caption or sfx or "Sound effect").strip()}
                if sfx and sfx.strip():
                    kw["sfx"] = sfx.strip()
                if music and music.strip():
                    kw["music"] = music.strip()
                if env and env.strip():
                    kw["env"] = env.strip()
                if speech and speech.strip():
                    kw["speech"] = speech.strip()
                if asr and asr.strip():
                    kw["asr"] = asr.strip()
                prompt_obj = self.model.compose_prompt(**kw)
            else:
                prompt_obj = tagged
            wav_t = self.model.generate(
                prompts=prompt_obj,
                num_steps=int(num_steps),
                guidance_scale=float(eval_cfg),
                sway_sampling_coef=float(sway_coef),
            )
            arr = _extract_dasheng_audio(wav_t)
            in_sr = 16000
        else:
            res = self.model.generate(
                tagged,
                eval_cfg=float(eval_cfg),
                stop_threshold=float(stop_threshold),
                min_stop_step=int(min_stop_step),
                seed=actual_seed,
            )
            arr = _extract_dasheng_audio(res["audio"] if isinstance(res, dict) else res)
            in_sr = int(res.get("sample_rate", 16000)) if isinstance(res, dict) else 16000

        up_arr, out_sr = upsample_with_harmonic_air(arr, in_sr=in_sr, target_sr=int(target_sr))
        return up_arr, out_sr, tagged, actual_seed, msg


DASHENG_STUDIO = XiaomiDashengStudio()
print("✅ Step 3 Complete: XiaomiDashengStudio initialized!")

In [ ]:
#@title Step 4: Launch MiDashengLM-Gen & Dasheng-AudioGen Gradio Studio (4 Tabs)
import os
import time
import gradio as gr
from IPython.display import clear_output

ENGINE_CHOICES = [
    "mispeech/midashenglm-gen (Qwen3-1.7B + 10-Step Flow Matching + Auto-Stop Head)",
    "mispeech/Dasheng-AudioGen (25-Step Sway-Sampled Unified Scene Generator)",
    "mispeech/Dasheng-AudioGen-Multilingual (Multilingual Speech + SFX + Music Scene Generator)",
]

SCENE_PRESETS = {
    "🪟 Pure UI Notification Chime": {
        "caption": "A crisp crystalline glass two-note ascending system notification chime.",
        "sfx": "crisp crystalline glass two-note ascending chime",
        "music": "",
        "env": "clean studio",
        "speech": "",
        "asr": "",
    },
    "🤖 Sci-Fi Talking OS Startup (Voice + Chime)": {
        "caption": "A calm futuristic female AI assistant greeting the user accompanied by a warm holographic startup chord.",
        "sfx": "futuristic holographic interface telemetry shimmer",
        "music": "warm ascending ambient synth pad chord",
        "env": "high-tech control room",
        "speech": "calm articulate futuristic female voice",
        "asr": "System online. All neural modules initialized.",
    },
    "🔌 Talking USB Device Connect": {
        "caption": "An upbeat hardware connection chime followed by a crisp AI confirmation voice.",
        "sfx": "upbeat two-tone ascending futuristic telemetry ping",
        "music": "",
        "env": "clean digital interface",
        "speech": "clear robotic assistant voice",
        "asr": "External hardware connected.",
    },
    "⚔️ Cyberpunk Sword Slash & Thunder": {
        "caption": "A high-energy plasma blade slash cutting through the air during a rainstorm with thunder.",
        "sfx": "sharp metallic plasma sword whoosh and electric crackle",
        "music": "dark tense synth bass pulse",
        "env": "heavy rain and distant thunder",
        "speech": "",
        "asr": "",
    },
}


def apply_scene_preset(preset_name):
    p = SCENE_PRESETS.get(preset_name, SCENE_PRESETS["🪟 Pure UI Notification Chime"])
    return p["caption"], p["sfx"], p["music"], p["env"], p["speech"], p["asr"]


def ui_dasheng_generate(
    engine_choice,
    caption,
    sfx,
    music,
    env,
    speech,
    asr,
    eval_cfg,
    stop_threshold,
    min_stop_step,
    num_steps,
    sway_coef,
    seed,
    target_sr,
    apply_mastering,
    trim_db,
    fade_out_ms,
):
    try:
        t0 = time.time()
        arr, sr, tagged_str, used_seed, load_msg = DASHENG_STUDIO.generate_scene(
            engine_choice=engine_choice,
            caption=caption,
            sfx=sfx,
            music=music,
            env=env,
            speech=speech,
            asr=asr,
            eval_cfg=float(eval_cfg),
            stop_threshold=float(stop_threshold),
            min_stop_step=int(min_stop_step),
            num_steps=int(num_steps),
            sway_coef=float(sway_coef),
            seed=int(seed),
            target_sr=int(target_sr),
        )
        if apply_mastering:
            arr = master_ui_sound(
                arr, sr, trim_silence=True, trim_db=float(trim_db),
                fade_in_ms=5.0, fade_out_ms=float(fade_out_ms),
            )
        out_wav = os.path.join(OUTPUT_DIR, f"dasheng_{int(time.time())}_s{used_seed}.wav")
        save_pcm16_wav(out_wav, arr, sr)
        spec_img = render_waveform_and_spectrogram(out_wav, title=f"Xiaomi Dasheng ({sr} Hz)")
        dur = len(arr) / float(sr)
        status = (
            f"✅ Generated **{dur:.2f}s** audio scene (`{sr} Hz` PCM) in **{time.time()-t0:.2f}s** (Seed `{used_seed}`).\n"
            f"- {load_msg}\n- **Formatted Tags**: `{tagged_str}`"
        )
        return out_wav, spec_img, tagged_str, status
    except Exception as e:
        return None, None, "", f"❌ Generation Error: {e}"


def ui_dasheng_windows_pack(
    engine_choice,
    scheme_name,
    theme_name,
    pack_mode,
    voice_persona,
    selected_events,
    eval_cfg,
    stop_threshold,
    min_stop_step,
    base_seed,
    target_sr,
    trim_db,
    fade_out_ms,
):
    try:
        if not selected_events:
            return None, None, None, "", "❌ Select at least one Windows system event."
        t0 = time.time()
        pack_dir = os.path.join(OUTPUT_DIR, f"dasheng_winpack_{int(time.time())}")
        os.makedirs(pack_dir, exist_ok=True)
        event_wav_map = {}
        first_wav = None
        seed_val = int(base_seed) if int(base_seed) >= 0 else 42
        talking_enabled = "Talking AI" in pack_mode

        for idx, ev_key in enumerate(selected_events):
            ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, WINDOWS_SYSTEM_EVENTS["SystemNotification"])
            sfx_desc = build_windows_event_prompt(ev_key, theme_name, "")
            spoken_line = TALKING_OS_PHRASES.get(ev_key, "") if talking_enabled else ""
            if spoken_line:
                cap = f"{sfx_desc} accompanied by a {voice_persona} saying '{spoken_line}'"
            else:
                cap = sfx_desc

            arr, sr, tagged, _, _ = DASHENG_STUDIO.generate_scene(
                engine_choice=engine_choice,
                caption=cap,
                sfx=ev_meta["contour"],
                music=SONIC_THEME_PRESETS.get(theme_name, ""),
                env="clean studio interface",
                speech=voice_persona if spoken_line else "",
                asr=spoken_line,
                eval_cfg=float(eval_cfg),
                stop_threshold=float(stop_threshold),
                min_stop_step=int(min_stop_step),
                seed=seed_val + idx,
                target_sr=int(target_sr),
            )
            max_dur = float(ev_meta["duration"]) + (1.5 if spoken_line else 0.2)
            mastered = master_ui_sound(
                arr, sr, target_duration_s=max_dur, trim_silence=True,
                trim_db=float(trim_db), fade_in_ms=5.0, fade_out_ms=float(fade_out_ms),
            )
            fname = f"{ev_key}.wav"
            fpath = os.path.join(pack_dir, fname)
            save_pcm16_wav(fpath, mastered, sr)
            event_wav_map[ev_key] = {"path": fpath, "filename": fname, "prompt": tagged}
            if first_wav is None:
                first_wav = fpath

        model_lbl = f"Xiaomi {engine_choice.split(' ')[0]}"
        zip_path = package_windows_sound_scheme_zip(
            scheme_name=scheme_name,
            event_wav_map=event_wav_map,
            theme_name=f"{theme_name} ({pack_mode})",
            model_label=model_lbl,
            output_dir=OUTPUT_DIR,
        )
        board_html = build_audition_soundboard_html(
            scheme_name=scheme_name,
            event_wav_map=event_wav_map,
            theme_name=f"{theme_name} ({pack_mode})",
            model_label=model_lbl,
            embed_base64=True,
        )
        spec_img = render_waveform_and_spectrogram(first_wav, title=f"Windows Event: {selected_events[0]}") if first_wav else None
        return zip_path, first_wav, spec_img, board_html, (
            f"✅ Built `{os.path.basename(zip_path)}` ({len(event_wav_map)} events, `{pack_mode}`) in **{time.time()-t0:.1f}s**!"
        )
    except Exception as e:
        return None, None, None, "", f"❌ Windows Pack Error: {e}"


def ui_dasheng_reroll_single_event(
    engine_choice,
    scheme_name,
    theme_name,
    pack_mode,
    voice_persona,
    ev_key,
    ev_custom_prompt,
    reroll_seed,
    pitch_semitones,
    eval_cfg,
    stop_threshold,
    min_stop_step,
    target_sr,
    trim_db,
    fade_out_ms,
):
    try:
        ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, WINDOWS_SYSTEM_EVENTS["SystemNotification"])
        sfx_desc = (ev_custom_prompt or "").strip() or build_windows_event_prompt(ev_key, theme_name, "")
        talking_enabled = "Talking AI" in pack_mode
        spoken_line = TALKING_OS_PHRASES.get(ev_key, "") if talking_enabled else ""
        cap = f"{sfx_desc} accompanied by a {voice_persona} saying '{spoken_line}'" if spoken_line else sfx_desc
        arr, sr, tagged, _, _ = DASHENG_STUDIO.generate_scene(
            engine_choice=engine_choice,
            caption=cap,
            sfx=ev_meta["contour"],
            music=SONIC_THEME_PRESETS.get(theme_name, ""),
            env="clean studio interface",
            speech=voice_persona if spoken_line else "",
            asr=spoken_line,
            eval_cfg=float(eval_cfg),
            stop_threshold=float(stop_threshold),
            min_stop_step=int(min_stop_step),
            seed=int(reroll_seed),
            target_sr=int(target_sr),
        )
        max_dur = float(ev_meta["duration"]) + (1.5 if spoken_line else 0.2)
        mastered = master_ui_sound(
            arr, sr, target_duration_s=max_dur, trim_silence=True,
            trim_db=float(trim_db), fade_in_ms=1.5, fade_out_ms=float(fade_out_ms),
            pitch_semitones=float(pitch_semitones),
        )
        return finalize_rerolled_event_in_pack(
            ev_key=ev_key,
            mastered_wav=mastered,
            sr=sr,
            ev_prompt=tagged,
            output_dir=OUTPUT_DIR,
            scheme_name=scheme_name,
            theme_name=f"{theme_name} ({pack_mode})",
            model_label=f"Xiaomi {engine_choice.split(' ')[0]}",
        )
    except Exception as e:
        return None, None, None, "", f"❌ Single-Event Re-Roll Error: {e}"


def ui_dasheng_batch(engine_choice, batch_lines, eval_cfg, stop_threshold, seed, target_sr):
    try:
        lines = [ln.strip() for ln in (batch_lines or "").splitlines() if ln.strip()]
        if not lines:
            return None, "❌ Enter at least one line."
        out_files = []
        for i, ln in enumerate(lines):
            arr, sr, _, _, _ = DASHENG_STUDIO.generate_scene(
                engine_choice=engine_choice, caption=ln, sfx=ln,
                eval_cfg=float(eval_cfg), stop_threshold=float(stop_threshold),
                seed=int(seed) + i, target_sr=int(target_sr),
            )
            arr = master_ui_sound(arr, sr)
            p = os.path.join(OUTPUT_DIR, f"dasheng_batch_{i+1:02d}_{int(time.time())}.wav")
            save_pcm16_wav(p, arr, sr)
            out_files.append(p)
        return out_files, f"✅ Generated **{len(out_files)}** files!"
    except Exception as e:
        return None, f"❌ Batch Error: {e}"


with gr.Blocks(title="Xiaomi MiDashengLM-Gen & Dasheng-AudioGen Studio", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "## 🤖 Xiaomi MiDashengLM-Gen & Dasheng-AudioGen — Unified Audio Scene, Auto-Stop SFX & Talking OS Pack Studio\n"
        "Combine `<|caption|>`, `<|sfx|>`, `<|music|>`, `<|env|>`, `<|speech|>`, and `<|asr|>` with automatic Stop-Head truncation!"
    )
    banner = gr.Markdown("⏳ Initializing Xiaomi Dasheng studio...")

    engine_dd = gr.Dropdown(
        choices=ENGINE_CHOICES,
        value=ENGINE_CHOICES[0],
        label="🧠 Xiaomi Unified Audio Backbone",
        info="Use 'midashenglm-gen' for 10-step Euler ODE + learned Stop Head, or 'Dasheng-AudioGen' for 25-step sway sampling.",
    )

    with gr.Tabs():
        with gr.Tab("🧩 1. Multi-Aspect Audio Scene & SFX Composer"):
            with gr.Row():
                with gr.Column(scale=3):
                    s_preset = gr.Dropdown(
                        choices=list(SCENE_PRESETS.keys()),
                        value=list(SCENE_PRESETS.keys())[0],
                        label="✨ Scene / Notification Preset",
                        info="Populate structured multi-aspect tags with 1 click.",
                    )
                    s_cap = gr.Textbox(
                        value=SCENE_PRESETS["🪟 Pure UI Notification Chime"]["caption"],
                        lines=2,
                        label="🎬 <|caption|> Overall Scene Description (Required)",
                        info="High-level summary of the sound effect or multi-modal scene.",
                    )
                    with gr.Row():
                        s_sfx = gr.Textbox(
                            value=SCENE_PRESETS["🪟 Pure UI Notification Chime"]["sfx"],
                            label="🔊 <|sfx|> Sound Effects Tag",
                            info="Specific transient/foley/chime sound effects.",
                        )
                        s_music = gr.Textbox(
                            value="",
                            label="🎵 <|music|> Musical Stinger / Pad Tag (Optional)",
                            info="Optional harmonic chord, instrument, or musical sting.",
                        )
                    with gr.Row():
                        s_env = gr.Textbox(
                            value="clean studio",
                            label="🌍 <|env|> Acoustic Environment Tag (Optional)",
                            info="E.g. 'clean studio', 'intimate room', 'cathedral hall'.",
                        )
                        s_speech = gr.Textbox(
                            value="",
                            label="🗣️ <|speech|> Voice Style Tag (Optional)",
                            info="E.g. 'calm futuristic female AI voice' (leave blank for pure SFX).",
                        )
                    s_asr = gr.Textbox(
                        value="",
                        label="💬 <|asr|> Spoken Transcript (Optional — Leave Blank for Pure SFX)",
                        info="Exact words spoken simultaneously with the SFX/music scene.",
                    )
                    with gr.Row():
                        s_cfg = gr.Slider(1.0, 7.0, value=2.5, step=0.5, label="🧭 CFG Scale (`eval_cfg`)", info="Default 2.0–2.5 for MiDashengLM-Gen; 5.0 for Dasheng-AudioGen.")
                        s_stop = gr.Slider(0.1, 0.9, value=0.5, step=0.05, label="🛑 Stop-Head Threshold (`stop_threshold`)", info="Learned binary classifier cutoff for auto-truncating short SFX.")
                        s_min_step = gr.Slider(1, 20, value=5, step=1, label="⏱️ Min Stop Steps (`min_stop_step`)", info="Minimum tokens before Stop Head can terminate.")
                    with gr.Row():
                        s_steps = gr.Slider(10, 50, value=25, step=5, label="⚡ Denoising Steps (Dasheng-AudioGen)", info="Used when Dasheng-AudioGen is selected.")
                        s_sway = gr.Slider(-1.0, 1.0, value=-1.0, step=0.1, label="🌊 Sway Sampling Coef", info="Default -1.0 (0 for linear).")
                        s_seed = gr.Number(value=42, precision=0, label="🎲 Seed", info="Random seed.")
                        s_sr = gr.Dropdown(choices=[44100, 48000, 16000], value=44100, label="📈 Output Sample Rate (Hz)", info="Polyphase upsampling + harmonic air exciter.")
                    with gr.Accordion("🛠️ Zero-Latency UI Mastering Controls", open=False):
                        with gr.Row():
                            s_master = gr.Checkbox(value=True, label="✨ Auto-Trim Leading Silence & Normalize", info="Zero-latency UI trim.")
                            s_trim = gr.Slider(-60.0, -25.0, value=-42.0, step=1.0, label="✂️ Silence Threshold (dB)", info="Trim threshold.")
                            s_fade = gr.Slider(20.0, 350.0, value=120.0, step=10.0, label="📉 Fade-Out (ms)", info="Release fade.")
                    s_btn = gr.Button("🚀 Synthesize Audio Scene / SFX", variant="primary")

                with gr.Column(scale=2):
                    s_wav = gr.Audio(label="🎧 Generated Audio Output", type="filepath")
                    s_spec = gr.Image(label="📊 Waveform & Spectrogram", type="filepath")
                    s_tags_preview = gr.Textbox(label="🏷️ Composed Multi-Tag Prompt String", interactive=False)
                    s_status = gr.Markdown()

            s_preset.change(fn=apply_scene_preset, inputs=[s_preset], outputs=[s_cap, s_sfx, s_music, s_env, s_speech, s_asr])
            s_btn.click(
                fn=ui_dasheng_generate,
                inputs=[engine_dd, s_cap, s_sfx, s_music, s_env, s_speech, s_asr, s_cfg, s_stop, s_min_step, s_steps, s_sway, s_seed, s_sr, s_master, s_trim, s_fade],
                outputs=[s_wav, s_spec, s_tags_preview, s_status],
            )

        with gr.Tab("🤖 2. Talking AI / Pure SFX Windows Sound Pack Builder"):
            gr.Markdown(
                "### 🪟 Windows 10/11 Sound Scheme Builder — Pure SFX or Talking Sci-Fi OS Assistant!\n"
                "Choose **Pure Instrumental/SFX Mode** for classic chimes, OR **Talking AI Assistant + SFX Mode** to blend "
                "a futuristic voice confirmation (*'System online'*, *'External hardware connected'*) with synchronized telemetry chimes!"
            )
            with gr.Row():
                with gr.Column(scale=3):
                    with gr.Row():
                        dw_scheme = gr.Textbox(value="Dasheng_SciFi_OS", label="🏷️ Windows Sound Scheme Name", info="Registered in Windows Sound Control Panel.")
                        dw_theme = gr.Dropdown(
                            choices=list(SONIC_THEME_PRESETS.keys()),
                            value="Sci-Fi Spaceship Bridge",
                            label="🎨 Sonic Theme Preset",
                            info="Underlying SFX and musical stinger aesthetic.",
                        )
                    with gr.Row():
                        dw_mode = gr.Radio(
                            choices=["Pure Instrumental / SFX Chimes Only", "Talking AI Assistant Voice + SFX Chimes"],
                            value="Pure Instrumental / SFX Chimes Only",
                            label="🎙️ Sound Pack Mode",
                            info="Pure chimes vs unified Speech + SFX notifications.",
                        )
                        dw_voice = gr.Textbox(
                            value="calm articulate futuristic female AI assistant voice",
                            label="🤖 AI Voice Persona (`<|speech|>` Tag when Talking Mode is active)",
                            info="Describe the voice timbre and emotion.",
                        )
                    dw_events = gr.CheckboxGroup(
                        choices=list(WINDOWS_SYSTEM_EVENTS.keys()),
                        value=DEFAULT_SELECTED_WIN_EVENTS,
                        label="🔔 Windows System Events to Generate",
                        info="Select events to include in the downloadable pack.",
                    )
                    with gr.Row():
                        dw_cfg = gr.Slider(1.0, 6.0, value=2.5, step=0.5, label="🧭 CFG Scale", info="Guidance scale.")
                        dw_stop = gr.Slider(0.2, 0.8, value=0.5, step=0.05, label="🛑 Stop Threshold", info="Auto-stop threshold.")
                        dw_min = gr.Slider(2, 15, value=5, step=1, label="⏱️ Min Stop Step", info="Minimum tokens.")
                        dw_seed = gr.Number(value=42, precision=0, label="🎲 Base Seed", info="Seed.")
                    with gr.Row():
                        dw_sr = gr.Dropdown(choices=[44100, 48000], value=44100, label="📈 Master Sample Rate (Hz)", info="16-bit PCM .wav sample rate.")
                        dw_trim = gr.Slider(-60.0, -25.0, value=-42.0, step=1.0, label="✂️ Trim (dB)", info="Zero-latency trim.")
                        dw_fade = gr.Slider(40.0, 300.0, value=120.0, step=10.0, label="📉 Fade-Out (ms)", info="Smooth release.")
                    dw_btn = gr.Button("🪟 Build Windows Sound Scheme (.zip + .ps1 + .reg)", variant="primary")
                    with gr.Accordion("🎯 Single-Event Re-Roll & Pitch Customizer (Update 1 Event Without Regenerating Pack)", open=False):
                        with gr.Row():
                            dw_rr_ev = gr.Dropdown(choices=list(WINDOWS_SYSTEM_EVENTS.keys()), value="DeviceConnect", label="🔔 Event to Re-Roll / Replace", info="Pick a single Windows event to regenerate.")
                            dw_rr_seed = gr.Number(value=-1, precision=0, label="🎲 New Seed (-1 = Random)", info="Fresh seed.")
                            dw_rr_pitch = gr.Slider(-6.0, 6.0, value=0.0, step=0.5, label="🎵 Pitch Transpose (Semitones)", info="Semitone shift.")
                        dw_rr_prompt = gr.Textbox(value="", placeholder="Optional custom prompt override for this event...", label="✍️ Single-Event Prompt Override (Optional)", info="Leave blank to use theme prompt.")
                        dw_rr_btn = gr.Button("🔄 Re-Roll Single Event & Update Pack")
                with gr.Column(scale=2):
                    dw_zip = gr.File(label="📦 Download Windows Sound Pack (.zip)")
                    dw_prev = gr.Audio(label="🎧 Sample Event Preview", type="filepath")
                    dw_spec = gr.Image(label="📊 Spectrogram", type="filepath")
                    dw_status = gr.Markdown()
            dw_board = gr.HTML(label="🎹 Interactive Windows Sound Scheme Audition Board")

            dw_btn.click(
                fn=ui_dasheng_windows_pack,
                inputs=[engine_dd, dw_scheme, dw_theme, dw_mode, dw_voice, dw_events, dw_cfg, dw_stop, dw_min, dw_seed, dw_sr, dw_trim, dw_fade],
                outputs=[dw_zip, dw_prev, dw_spec, dw_board, dw_status],
            )
            dw_rr_btn.click(
                fn=ui_dasheng_reroll_single_event,
                inputs=[engine_dd, dw_scheme, dw_theme, dw_mode, dw_voice, dw_rr_ev, dw_rr_prompt, dw_rr_seed, dw_rr_pitch, dw_cfg, dw_stop, dw_min, dw_sr, dw_trim, dw_fade],
                outputs=[dw_zip, dw_prev, dw_spec, dw_board, dw_status],
            )

        with gr.Tab("📦 3. Batch Scene Generator"):
            with gr.Row():
                with gr.Column():
                    db_lines = gr.Textbox(
                        value=(
                            "Crisp glass two-note ascending notification chime\n"
                            "Warm wooden marimba completion flourish\n"
                            "Futuristic holographic telemetry double beep"
                        ),
                        lines=6,
                        label="📋 One Scene Description per Line",
                        info="Each line is formatted and generated in sequence.",
                    )
                    with gr.Row():
                        db_cfg = gr.Slider(1.0, 6.0, value=2.5, step=0.5, label="🧭 CFG", info="Guidance scale.")
                        db_stop = gr.Slider(0.2, 0.8, value=0.5, step=0.05, label="🛑 Stop Threshold", info="Auto-stop cutoff.")
                        db_seed = gr.Number(value=42, precision=0, label="🎲 Seed", info="Seed.")
                        db_sr = gr.Dropdown(choices=[44100, 48000], value=44100, label="📈 Sample Rate", info="Output sample rate.")
                    db_btn = gr.Button("📦 Run Batch Generation", variant="primary")
                with gr.Column():
                    db_files = gr.File(label="📂 Generated Batch WAVs", file_count="multiple")
                    db_status = gr.Markdown()

            db_btn.click(fn=ui_dasheng_batch, inputs=[engine_dd, db_lines, db_cfg, db_stop, db_seed, db_sr], outputs=[db_files, db_status])

    demo.load(
        fn=lambda: f"🟢 **Xiaomi MiDashengLM-Gen & Dasheng-AudioGen Studio Ready** on `{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}`!",
        inputs=None,
        outputs=[banner],
    )

clear_output()
demo.queue(default_concurrency_limit=1)
demo.launch(share=True, inline=False, show_error=True)

In [ ]:
#@title Step 5: Keep-Alive Helper & GPU VRAM Monitor
import time
import torch

KEEP_ALIVE_MINUTES = 0 #@param {type:"integer"}

try:
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated(0) / (1024 ** 3)
        resv = torch.cuda.memory_reserved(0) / (1024 ** 3)
        print(f"📊 GPU VRAM Status — Allocated: {alloc:.2f} GB | Reserved: {resv:.2f} GB")
    if KEEP_ALIVE_MINUTES > 0:
        print(f"⏳ Keeping Gradio session alive for {KEEP_ALIVE_MINUTES} minutes...")
        for m in range(KEEP_ALIVE_MINUTES):
            time.sleep(60)
            print(f"  ✓ Active: {m + 1}/{KEEP_ALIVE_MINUTES} min elapsed")
    else:
        print("ℹ️ Keep-alive loop skipped (KEEP_ALIVE_MINUTES=0). Set > 0 if running unattended.")
except KeyboardInterrupt:
    print("⏹️ Keep-alive stopped by user.")
except Exception as e:
    print(f"Keep-alive warning: {e}")

In [ ]:
#@title Step 6: Quick Test — Auto-Stop SFX Scene Generation
import os
from IPython.display import Audio, FileLink, display

QUICKTEST_ENGINE = "mispeech/midashenglm-gen (Qwen3-1.7B + 10-Step Flow Matching + Auto-Stop Head)" #@param ["mispeech/midashenglm-gen (Qwen3-1.7B + 10-Step Flow Matching + Auto-Stop Head)", "mispeech/Dasheng-AudioGen (25-Step Sway-Sampled Unified Scene Generator)"]
QUICKTEST_CAPTION = "A crisp crystalline glass two-note ascending system notification chime." #@param {type:"string"}
QUICKTEST_SFX = "crisp crystalline glass two-note ascending chime" #@param {type:"string"}

try:
    wav_path, _, tags_used, msg = ui_dasheng_generate(
        engine_choice=QUICKTEST_ENGINE,
        caption=QUICKTEST_CAPTION,
        sfx=QUICKTEST_SFX,
        music="",
        env="clean studio",
        speech="",
        asr="",
        eval_cfg=2.5,
        stop_threshold=0.5,
        min_stop_step=5,
        num_steps=25,
        sway_coef=-1.0,
        seed=42,
        target_sr=44100,
        apply_mastering=True,
        trim_db=-42.0,
        fade_out_ms=120.0,
    )
    print(msg)
    if wav_path and os.path.exists(wav_path):
        display(Audio(filename=wav_path))
        display(FileLink(wav_path))
except Exception as e:
    print(f"Quick test error: {e}")

In [ ]:
#@title Step 7: Headless Batch Windows Sound Scheme Generator
import os
from tqdm.auto import tqdm
from IPython.display import FileLink, display

BATCH_SCHEME_NAME = "Xiaomi_Dasheng_OS" #@param {type:"string"}
BATCH_THEME = "Sci-Fi Spaceship Bridge" #@param ["Frutiger Aero Crystal Glass", "Cyberpunk Neon HUD", "Cozy Wooden Kalimba & Mallet", "Sci-Fi Spaceship Bridge", "Retro 16-Bit Chiptune Arcade", "Warm Lo-Fi Analog Tape Synth", "Minimalist Tactile Micro-Foley", "Dark Fantasy Crystal & Harp"]
BATCH_SEED = 42 #@param {type:"integer"}

try:
    event_map = {}
    pack_dir = os.path.join(OUTPUT_DIR, f"batch_dasheng_winpack_{BATCH_SEED}")
    os.makedirs(pack_dir, exist_ok=True)
    for idx, ev_key in enumerate(tqdm(DEFAULT_SELECTED_WIN_EVENTS, desc="MiDashengLM-Gen Windows Pack")):
        try:
            ev_meta = WINDOWS_SYSTEM_EVENTS[ev_key]
            prompt = build_windows_event_prompt(ev_key, BATCH_THEME, "")
            arr, sr, tagged, _, _ = DASHENG_STUDIO.generate_scene(
                engine_choice="mispeech/midashenglm-gen",
                caption=prompt,
                sfx=ev_meta["contour"],
                music=SONIC_THEME_PRESETS[BATCH_THEME],
                env="clean studio",
                seed=BATCH_SEED + idx,
                target_sr=44100,
            )
            mastered = master_ui_sound(arr, sr, target_duration_s=float(ev_meta["duration"]))
            fpath = os.path.join(pack_dir, f"{ev_key}.wav")
            save_pcm16_wav(fpath, mastered, sr)
            event_map[ev_key] = {"path": fpath, "filename": f"{ev_key}.wav", "prompt": tagged}
        except Exception as item_e:
            print(f"⚠️ Event {ev_key} skipped: {item_e}")

    if event_map:
        zip_out = package_windows_sound_scheme_zip(BATCH_SCHEME_NAME, event_map, BATCH_THEME, "Xiaomi MiDashengLM-Gen", OUTPUT_DIR)
        print(f"✅ Packaged Windows Sound Scheme: {zip_out}")
        display(FileLink(zip_out))
except Exception as e:
    print(f"Batch Windows Pack error: {e}")